# Chapter 2 — Exploratory Data Analysis
**MADT6004 · Brew Lab BKK case**

Before any model you build a feel for the data. EDA covers:
- Shape, types, missing values
- Distributions (histograms, boxplots)
- Relationships (scatter, group means)
- Outliers and oddities

In this notebook you'll EDA the Brew Lab transaction data to surface anything strange before you start modeling.


## 0. Bootstrap (Colab + local)

In [ ]:
# Bootstrap — make sure brewlab.db is available, both locally and in Colab.
import os
DB_CANDIDATES = [
    "../../Integrated Data Analytics Exercise/data/brewlab.db",
    "MADT6004/Integrated Data Analytics Exercise/data/brewlab.db",
]
DB_PATH = next((p for p in DB_CANDIDATES if os.path.exists(p)), None)
if DB_PATH is None:
    if not os.path.exists("MADT6004"):
        os.system("git clone -q https://github.com/thanachart/MADT6004.git")
    os.system("pip install -q -r 'MADT6004/Integrated Data Analytics Exercise/requirements.txt'")
    DB_PATH = "MADT6004/Integrated Data Analytics Exercise/data/brewlab.db"
print("DB:", DB_PATH)


## 1. Setup

In [ ]:
import sqlite3
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 100

conn = sqlite3.connect(DB_PATH)
print("Tables:", [r[0] for r in conn.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()])


## 2. The shape of the transactions table
Pull a sample, check dtypes, and look for missing values.

In [ ]:
tx = pd.read_sql("SELECT * FROM transactions", conn)
tx["datetime"] = pd.to_datetime(tx["datetime"])

print("Shape:", tx.shape)
print()
print("Dtypes:")
print(tx.dtypes)
print()
print("Missing values per column:")
print(tx.isna().sum())


## 3. Distribution of order totals
A histogram of `total` shows the typical ticket and the long tail.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].hist(tx["total"], bins=50, color="#0891B2", edgecolor="white")
ax[0].set_title("Order total — histogram")
ax[0].set_xlabel("THB")

ax[1].boxplot(tx["total"], vert=False)
ax[1].set_title("Order total — boxplot")
ax[1].set_xlabel("THB")
plt.tight_layout(); plt.show()

print(tx["total"].describe())


## 4. Daily revenue over time
Aggregate to daily level and plot. Look for trend, seasonality, and gaps.

In [ ]:
daily = tx.set_index("datetime").resample("D")["total"].sum()
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(daily.index, daily.values, color="#0891B2", lw=0.8)
ax.set_title("Daily revenue across all branches")
ax.set_ylabel("Revenue (THB)")
plt.tight_layout(); plt.show()


## 5. Revenue spread across branches
A boxplot of daily revenue per branch reveals which branches are big and which are volatile.

In [ ]:
daily_branch = (tx.assign(d=tx["datetime"].dt.date)
                  .groupby(["branch_id", "d"])["total"].sum().reset_index())
branches = pd.read_sql("SELECT branch_id, name FROM branches", conn)
daily_branch = daily_branch.merge(branches, on="branch_id")

order = daily_branch.groupby("name")["total"].median().sort_values().index
fig, ax = plt.subplots(figsize=(11, 5))
sns.boxplot(data=daily_branch, x="name", y="total", order=order, ax=ax,
            color="#0891B2")
ax.set_xlabel(""); ax.set_ylabel("Daily revenue (THB)")
ax.set_title("Daily revenue distribution by branch")
plt.xticks(rotation=35, ha="right")
plt.tight_layout(); plt.show()


## Discussion prompts
1. Which branch has the most volatile daily revenue? What might explain it?
2. Are there outliers in `total`? Do you treat them, or are they real big orders?
3. EDA usually surfaces *one* surprise that changes how you'd model. What surprised you?
